<a href="https://colab.research.google.com/github/nyp-sit/iti121-2026s2/blob/main/L4/03_finetune_with_data_augmentation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Notebook 3 — Data Augmentation + Fine-tuning

In Notebook 2 we saw that unfreezing more layers makes the model fit the training data very well — sometimes *too* well (overfitting), because we only have ~2,900 training images.

**Data augmentation** is the standard remedy: every time an image is loaded we apply a **random** transformation (crop, flip, rotation, colour change, …). The network almost never sees exactly the same picture twice, so it is forced to learn features that are **robust** to these changes instead of memorising pixels.

### Main ideas in this notebook
1. Building an augmentation pipeline with `torchvision.transforms`.
2. Augment the **training set only** — validation/test images stay deterministic.
3. Compare fine-tuning (`layer4` unfrozen) **with vs without** augmentation.
4. Why augmentation is **incompatible with the feature-caching trick** from Notebook 1.

## 1. Setup, data and helpers

In [ ]:
import os, time, copy, random
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms, models
from torchvision.datasets.utils import download_and_extract_archive

SEED = 42
def set_seed(seed=SEED):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
set_seed()

# Pick the fastest available device: NVIDIA GPU -> Apple Silicon GPU -> CPU
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
print("PyTorch", torch.__version__, "| device:", device)

In [ ]:
DATA_URL  = "https://storage.googleapis.com/download.tensorflow.org/example_images/flower_photos.tgz"
DATA_ROOT = "data"
DATA_DIR  = os.path.join(DATA_ROOT, "flower_photos")

if not os.path.isdir(DATA_DIR):
    download_and_extract_archive(DATA_URL, download_root=DATA_ROOT)   # ~220 MB, only downloaded once

# One sub-folder per class -> ImageFolder labels the images automatically
class_names = sorted(d for d in os.listdir(DATA_DIR) if os.path.isdir(os.path.join(DATA_DIR, d)))
for c in class_names:
    print(f"{c:12s} {len(os.listdir(os.path.join(DATA_DIR, c)))} images")

In [ ]:
# ResNet18 was trained on ImageNet, so we must feed it images prepared the SAME way:
# 224x224 pixels, normalised with the ImageNet channel mean and standard deviation.
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

eval_tf = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

## 2. The augmentation pipeline

| Transform | What it simulates |
|---|---|
| `RandomResizedCrop(224, scale=(0.6, 1.0))` | different zoom levels / framing of the flower |
| `RandomHorizontalFlip()` | a mirror image is still the same flower |
| `RandomRotation(15)` | camera tilted slightly |
| `ColorJitter(brightness, contrast, saturation)` | different lighting and cameras |

Choose augmentations that produce images which are **still realistic and still the same class**. E.g. a vertical flip is fine for flowers seen from above but would be wrong for digits like 6 / 9; a big hue shift could turn a yellow sunflower purple, so we leave hue alone.

The final steps (`ToTensor`, `Normalize`) are the same as before — the pre-trained network still needs ImageNet-normalised input.

In [ ]:
aug_tf = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.6, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

### Seeing augmentation in action
Each call to the transform gives a **different** random version of the same photo.

In [ ]:
from PIL import Image
mean, std = torch.tensor(IMAGENET_MEAN).view(3,1,1), torch.tensor(IMAGENET_STD).view(3,1,1)
denorm = lambda t: (t * std + mean).clamp(0, 1).permute(1, 2, 0)

raw = datasets.ImageFolder(DATA_DIR)          # no transform -> PIL images
for i in np.random.RandomState(1).choice(len(raw), 2, replace=False):
    img, label = raw[i]
    fig, axes = plt.subplots(1, 7, figsize=(17, 2.8))
    axes[0].imshow(denorm(eval_tf(img))); axes[0].set_title("original (eval_tf)")
    for ax in axes[1:]:
        ax.imshow(denorm(aug_tf(img))); ax.set_title("augmented")
    for ax in axes: ax.axis("off")
    fig.suptitle(class_names[label]); plt.tight_layout(); plt.show()

## 3. Datasets: augment the training set only

* **Training set** → `aug_tf` (random, a new version every epoch)
* **Validation set** → `eval_tf` (deterministic) — we want to measure performance on *real*, unmodified images, and we want the measurement to be the same every time.

We create two training sets so we can compare *without* and *with* augmentation on the identical split.

In [ ]:
def make_datasets(train_tf, val_tf, val_frac=0.2, seed=SEED):
    """Same fixed 80/20 split in every notebook, so results are comparable."""
    n = len(datasets.ImageFolder(DATA_DIR))
    idx = np.random.RandomState(seed).permutation(n)
    n_val = int(val_frac * n)
    val_idx, train_idx = idx[:n_val], idx[n_val:]
    # Two ImageFolder objects over the same files so train/val can use different transforms
    train_ds = Subset(datasets.ImageFolder(DATA_DIR, transform=train_tf), train_idx)
    val_ds   = Subset(datasets.ImageFolder(DATA_DIR, transform=val_tf),   val_idx)
    return train_ds, val_ds

BATCH_SIZE  = 32
NUM_WORKERS = 2          # set to 0 if you get DataLoader errors (e.g. on Windows)
PIN = device.type == "cuda"

def make_loaders(train_ds, val_ds, shuffle_train=True):
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=shuffle_train,
                              num_workers=NUM_WORKERS, pin_memory=PIN)
    val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                              num_workers=NUM_WORKERS, pin_memory=PIN)
    return train_loader, val_loader

train_plain, val_ds = make_datasets(train_tf=eval_tf, val_tf=eval_tf)
train_aug,   _      = make_datasets(train_tf=aug_tf,  val_tf=eval_tf)

loader_plain, val_loader = make_loaders(train_plain, val_ds)
loader_aug,   _          = make_loaders(train_aug,   val_ds)
print(f"train: {len(train_aug)} | val: {len(val_ds)}")

In [ ]:
def set_train_mode(model):
    """model.train(), but keep BatchNorm layers of FROZEN blocks in eval mode.

    In train mode BatchNorm keeps updating its running mean/variance even when its
    weights are frozen - that would silently change the 'frozen' features.
    """
    model.train()
    for m in model.modules():
        if isinstance(m, nn.BatchNorm2d) and not any(p.requires_grad for p in m.parameters()):
            m.eval()

def run_epoch(model, loader, criterion, optimizer=None):
    """One pass over `loader`. Trains if an optimizer is given, otherwise just evaluates."""
    total_loss, correct, n = 0.0, 0, 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        out = model(x)
        loss = criterion(out, y)
        if optimizer is not None:
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
        total_loss += loss.item() * x.size(0)
        correct    += (out.argmax(1) == y).sum().item()
        n          += x.size(0)
    return total_loss / n, correct / n

def fit(model, train_loader, val_loader, optimizer, epochs, scheduler=None, verbose=True):
    """Standard training loop. Returns the history and restores the best weights (by val acc)."""
    criterion = nn.CrossEntropyLoss()
    hist = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": [], "epoch_time": []}
    best_acc, best_state = -1.0, None
    for ep in range(1, epochs + 1):
        t0 = time.time()
        set_train_mode(model)
        tl, ta = run_epoch(model, train_loader, criterion, optimizer)
        model.eval()
        with torch.no_grad():
            vl, va = run_epoch(model, val_loader, criterion)
        if scheduler is not None:
            scheduler.step()
        dt = time.time() - t0
        for k, v in zip(hist, (tl, ta, vl, va, dt)):
            hist[k].append(v)
        if va > best_acc:
            best_acc, best_state = va, copy.deepcopy(model.state_dict())
        if verbose:
            print(f"epoch {ep:2d}/{epochs} | train loss {tl:.3f} acc {ta:.3f} | "
                  f"val loss {vl:.3f} acc {va:.3f} | {dt:.1f}s")
    model.load_state_dict(best_state)
    hist["best_val_acc"] = best_acc
    return hist

def count_params(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"trainable params: {trainable:,} / {total:,}  ({100*trainable/total:.2f}%)")
    return trainable

### Why we can't cache features any more

In Notebook 1 we ran every image through the frozen base **once** and re-used the features. With augmentation, the input image is different in every epoch, so its features are different too. Caching would freeze one random version of each image and throw away the benefit of augmentation.

➡ **Augmentation means a full forward pass through the base every epoch** (and, with unfrozen layers, a backward pass too). It costs more compute — that is the price for better generalisation.

## 4. Model: ResNet18 with `layer4` unfrozen

We reuse the best strategy from Notebook 2: fine-tune `layer4` + new `fc`, with discriminative learning rates. A **cosine learning-rate schedule** gradually lowers the learning rate, which helps the model settle at the end of training.

Because augmented data is "harder" and more varied, it usually needs **more epochs** to converge than plain data, so we train a bit longer than in Notebook 2.

In [ ]:
HEAD_LR, BASE_LR = 1e-3, 1e-4
EPOCHS = 10

def build_finetune_model(num_classes):
    model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
    for p in model.parameters():
        p.requires_grad = False
    for p in model.layer4.parameters():
        p.requires_grad = True
    model.fc = nn.Linear(model.fc.in_features, num_classes)
    model = model.to(device)
    opt = optim.Adam([
        {"params": model.layer4.parameters(), "lr": BASE_LR},
        {"params": model.fc.parameters(),     "lr": HEAD_LR},
    ])
    sched = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)
    return model, opt, sched

## 5. Experiment 1 — fine-tuning **without** augmentation

In [ ]:
set_seed()
model_plain, opt, sched = build_finetune_model(len(class_names))
hist_plain = fit(model_plain, loader_plain, val_loader, opt, EPOCHS, scheduler=sched)

## 6. Experiment 2 — fine-tuning **with** augmentation

In [ ]:
set_seed()
model_aug, opt, sched = build_finetune_model(len(class_names))
hist_aug = fit(model_aug, loader_aug, val_loader, opt, EPOCHS, scheduler=sched)

## 7. Comparison

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.2))
for name, h, c in [("no augmentation", hist_plain, "tab:red"), ("with augmentation", hist_aug, "tab:green")]:
    ep = range(1, len(h["val_acc"]) + 1)
    axes[0].plot(ep, h["train_acc"], "--", color=c, label=f"{name} — train")
    axes[0].plot(ep, h["val_acc"],   "-",  color=c, marker="o", label=f"{name} — val")
    axes[1].plot(ep, h["train_loss"], "--", color=c, label=f"{name} — train")
    axes[1].plot(ep, h["val_loss"],   "-",  color=c, marker="o", label=f"{name} — val")
axes[0].set_title("Accuracy"); axes[1].set_title("Loss")
for ax in axes: ax.set_xlabel("epoch"); ax.grid(alpha=.3); ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

print(f"{'':20s} {'final train acc':>16s} {'best val acc':>13s} {'gap':>7s} {'s/epoch':>8s}")
for name, h in [("no augmentation", hist_plain), ("with augmentation", hist_aug)]:
    gap = h["train_acc"][-1] - h["val_acc"][-1]
    print(f"{name:20s} {h['train_acc'][-1]:>16.3f} {h['best_val_acc']:>13.3f} {gap:>7.3f} "
          f"{np.mean(h['epoch_time']):>8.1f}")

### What to look for

* **Without augmentation** — training accuracy shoots to ~100 % and training loss goes to almost zero, while validation loss stops improving or even *rises*: classic **overfitting**.
* **With augmentation** — training accuracy is *lower* (the task is harder: the model sees new variations every epoch), but the **gap between train and val is much smaller**, and validation accuracy is typically as good or better.
* A lower training accuracy is **not** a bad sign here: it means the model cannot simply memorise the images.

> On a small, fairly clean dataset like this the val-accuracy improvement may be a modest 1–3 %. The bigger, more reliable effect is the reduced overfitting, which matters more as you train longer or unfreeze more layers.

## 8. Where does the final model still make mistakes?
A confusion matrix shows which classes get mixed up.

In [ ]:
@torch.no_grad()
def predict(model, loader):
    model.eval(); preds, labels = [], []
    for x, y in loader:
        preds.append(model(x.to(device)).argmax(1).cpu()); labels.append(y)
    return torch.cat(preds).numpy(), torch.cat(labels).numpy()

best_model = model_aug if hist_aug["best_val_acc"] >= hist_plain["best_val_acc"] else model_plain
preds, labels = predict(best_model, val_loader)

C = len(class_names)
cm = np.zeros((C, C), dtype=int)
for t, p in zip(labels, preds): cm[t, p] += 1

fig, ax = plt.subplots(figsize=(5.5, 5))
ax.imshow(cm, cmap="Blues")
ax.set_xticks(range(C)); ax.set_xticklabels(class_names, rotation=45, ha="right")
ax.set_yticks(range(C)); ax.set_yticklabels(class_names)
ax.set_xlabel("predicted"); ax.set_ylabel("true")
for i in range(C):
    for j in range(C):
        ax.text(j, i, cm[i, j], ha="center", va="center", color="white" if cm[i, j] > cm.max()/2 else "black")
ax.set_title(f"Validation accuracy: {(preds == labels).mean():.3f}")
plt.tight_layout(); plt.show()

In [ ]:
# A few misclassified validation images
wrong = np.where(preds != labels)[0][:8]
if len(wrong):
    fig, axes = plt.subplots(1, len(wrong), figsize=(2.3*len(wrong), 2.8))
    axes = np.atleast_1d(axes)
    for ax, i in zip(axes, wrong):
        img, _ = val_ds[i]
        ax.imshow(denorm(img)); ax.axis("off")
        ax.set_title(f"true: {class_names[labels[i]]}\npred: {class_names[preds[i]]}", fontsize=9)
    plt.tight_layout(); plt.show()

In [ ]:
torch.save(best_model.state_dict(), "resnet18_flowers_finetuned.pt")
print("saved best model weights")

## 9. Summary of the three notebooks

| Notebook | Technique | Trainable | Feature caching? | Main benefit | Main cost |
|---|---|---|---|---|---|
| 1 | Freeze base, replace `fc` | `fc` only | ✅ | very fast, strong baseline, easy hyper-parameter search | features not adapted to flowers |
| 2 | Unfreeze last 1–2 conv blocks | `layer4` (+`layer3`) + `fc` | ❌ | higher accuracy, features adapt to the task | slower, risk of overfitting |
| 3 | Fine-tuning + data augmentation | `layer4` + `fc` | ❌ | better generalisation, less overfitting | more compute / epochs |

**Practical recipe for a new image-classification task**
1. Start with a frozen pre-trained base + new head (use cached features to quickly tune the head).
2. Unfreeze the last block(s) with a small learning rate (discriminative learning rates).
3. Add data augmentation on the training set whenever you fine-tune more layers or train for longer.
4. Always evaluate on un-augmented validation data, and watch the train–val gap.